# Tratamento de Base de Reclamações - Consumidor.gov.br

Pipeline de ET (Extract, Transform) para consolidar arquivos mensais de reclamações,
padronizar colunas, tratar valores nulos e ajustar tipos de dados para posterior carga em
banco de dados relacional (PostgreSQL).

O **Load** não faz parte deste notebook: a carga na tabela é feita separadamente, via `pgAdmin`
e script SQL dedicado (`sql/criacao_tabela.sql`).

**Etapas do notebook:**
1. Carregamento e exploração inicial dos dados
2. Consolidação de múltiplos arquivos mensais
3. Análise de qualidade dos dados (nulos, categorias)
4. Tratamento e conversão de tipos
5. Exportação do dataset tratado (pronto para carga no banco)


## 1. Carregamento dos dados

Leitura exploratória do primeiro arquivo, apenas para conhecer o dataset.

In [6]:
import pandas as pd
from pathlib import Path

data = Path('../data')

In [7]:
df = pd.read_csv(data/'2026-01.csv', sep=";")

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 287036 entries, 0 to 287035
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Região                  287036 non-null  str    
 1   UF                      287036 non-null  str    
 2   Cidade                  287036 non-null  str    
 3   Sexo                    287032 non-null  str    
 4   Faixa Etária            287036 non-null  str    
 5   Data Finalização        287036 non-null  str    
 6   Tempo Resposta          264922 non-null  float64
 7   Nome Fantasia           287036 non-null  str    
 8   Segmento de Mercado     287036 non-null  str    
 9   Área                    287036 non-null  str    
 10  Assunto                 287036 non-null  str    
 11  Grupo Problema          287036 non-null  str    
 12  Problema                287036 non-null  str    
 13  Como Comprou Contratou  287036 non-null  str    
 14  Procurou Empresa        287036 

## 2. Consolidação dos arquivos mensais

Todos os arquivos `.csv` da pasta `data/` são lidos, padronizados (nomes de coluna sem espaços) e concatenados em um único DataFrame.

In [9]:
dfs = []

for arq in data.glob('*.csv'):
    df_temp = pd.read_csv(arq, sep=';', encoding='utf-8-sig')
    df_temp.columns = df_temp.columns.str.strip()
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

### 3. Qualidade dos dados

Investigação de valores nulos e categorias inconsistentes antes de decidir como tratar cada coluna.

In [10]:
df.isnull().sum()

Região                          0
UF                              0
Cidade                          0
Sexo                          107
Faixa Etária                    0
Data Finalização                0
Tempo Resposta             108706
Nome Fantasia                   0
Segmento de Mercado             0
Área                            0
Assunto                         0
Grupo Problema                  0
Problema                        0
Como Comprou Contratou          0
Procurou Empresa                0
Respondida                      0
Situação                        0
Avaliação Reclamação            4
Nota do Consumidor        1382231
dtype: int64

Há dados nulos em `sexo`, `tempo de resposta`, `avaliação de reclamação` e `nota do consumidor`. Cada caso separar investigado separadamente abaixo:

In [11]:
#Coluna Tempo Resposta está nula, porque não houve reposta das empresas.
df[df['Tempo Resposta'].isnull()]['Respondida'].value_counts()

Respondida
N    108706
Name: count, dtype: int64

In [12]:
#Houve resposta da empresa, porém o consumidor não fez avaliação
df[df['Avaliação Reclamação'].isnull()]['Respondida'].value_counts()

Respondida
S    4
Name: count, dtype: int64

In [13]:
#S = Houve reposta da empresa, porém o consumidor não deu nota - nula por "consumidor não avaliou"
#N = A empresa nem respondeu - nula por "não tinha o que avaliar"
df[df['Nota do Consumidor'].isnull()]['Respondida'].value_counts()

Respondida
S    1297977
N      84254
Name: count, dtype: int64

## 4. Tratamento de dados


Com a causa dos nulos identificada, aplicam-se as correções:
- `Sexo` nulo → `'Não informado'`
- Colunas de data → convertidas para `datetime`
- Colunas numéricas inteiras → convertidas para `Int64` (nullable), preservando os nulos legítimos

In [14]:
df['Sexo'] = df['Sexo'].fillna('Não informado')

In [15]:
df['Sexo'].value_counts(dropna=False)

Sexo
M                1124897
F                 887356
O                   1793
Não informado        107
Name: count, dtype: int64

In [16]:
#transformado as colunas em data
df['Data Finalização'] = pd.to_datetime(df['Data Finalização'], errors='coerce', dayfirst=True)


In [17]:
#transformando as colunas inteiro
df['Tempo Resposta'] = df['Tempo Resposta'].astype('Int64')
df['Nota do Consumidor'] = df['Nota do Consumidor'].astype('Int64')

## 5. Exportação do dataset tratado

o resultado final é salvo em `data/` e fica pronto para ser carregado na tabela do PostegreSQL(via pgadmin ou script de carga).

In [18]:
#Salvar
df.to_csv(data/'dados_consumidor_tratado.csv', index=False, encoding='utf-8')

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2014153 entries, 0 to 2014152
Data columns (total 19 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   Região                  str           
 1   UF                      str           
 2   Cidade                  str           
 3   Sexo                    str           
 4   Faixa Etária            str           
 5   Data Finalização        datetime64[us]
 6   Tempo Resposta          Int64         
 7   Nome Fantasia           str           
 8   Segmento de Mercado     str           
 9   Área                    str           
 10  Assunto                 str           
 11  Grupo Problema          str           
 12  Problema                str           
 13  Como Comprou Contratou  str           
 14  Procurou Empresa        str           
 15  Respondida              str           
 16  Situação                str           
 17  Avaliação Reclamação    str           
 18  Nota do Consu